# Notebook 3: In-Context Learning (ICL) — Few-Shot Financial QA
---

**Objective**: Improve model performance by adding curated financial QA examples to the prompt (In-Context Learning). Then re-evaluate to quantify the improvement over the baseline.

**DLI Course Mapping**: Learning Objective 3 — "In-Context Learning with few-shot examples"

**What you'll do**:
1. Select high-quality ICL examples from the training set
2. Re-query NIM with few-shot prompts
3. Re-evaluate with the same metrics pipeline
4. Compare ICL vs Baseline quantitatively

In [ ]:
# ============================================================
# Setup & Imports
# ============================================================
import os
import sys
import json
import time
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from utils import (
    NIMInferenceClient,
    LLMJudge,
    load_financebench,
    format_finance_prompt,
    set_seed,
    save_results,
    load_results,
    compute_exact_match,
    compute_f1_score,
    print_metrics_summary,
    log_metrics_to_mlflow,
    RESULTS_DIR,
)

set_seed(42)

## 1. Select ICL Examples

> **From DLI Course**: The quality of few-shot examples matters more than quantity. Choose diverse examples that cover different types of financial questions (revenue, margins, YoY comparisons, etc.).

In [ ]:
# ============================================================
# Load Training Data for ICL Example Selection
# ============================================================

train_df = load_results("train_split.csv")
test_df = load_results("test_split.csv")
baseline_df = load_results("baseline_responses.csv")

print(f"Train examples available for ICL: {len(train_df)}")
print(f"Test examples to evaluate: {len(test_df)}")

In [ ]:
# ============================================================
# Curate ICL Examples — From NVIDIA DLI Course Lab
# ============================================================
# Strategy: Select diverse examples covering different financial concepts
# We pick 5 examples that demonstrate different answer patterns

def select_icl_examples(train_df, n_examples=5, seed=42):
    """
    Select diverse ICL examples from the training set.
    From NVIDIA DLI course lab — few-shot example curation.
    """
    # Shuffle with seed for reproducibility
    candidates = train_df.sample(frac=1, random_state=seed).reset_index(drop=True)

    # Select examples with varying answer lengths for diversity
    candidates["answer_len"] = candidates["answer"].str.len()
    candidates = candidates.sort_values("answer_len").reset_index(drop=True)

    # Pick evenly spaced examples across the answer-length distribution
    indices = np.linspace(0, len(candidates) - 1, n_examples, dtype=int)
    selected = candidates.iloc[indices]

    examples = []
    for _, row in selected.iterrows():
        examples.append({
            "question": row["question"],
            "answer": row["answer"],
            "context": str(row.get("context", ""))[:500],
            "evidence": str(row.get("evidence", ""))[:500],
        })

    return examples

# Select 5 diverse ICL examples
icl_examples = select_icl_examples(train_df, n_examples=5, seed=42)

print(f"Selected {len(icl_examples)} ICL examples:\n")
for i, ex in enumerate(icl_examples):
    print(f"  Example {i+1}: {ex['question'][:100]}...")
    print(f"  Answer:    {ex['answer'][:100]}...\n")

## 2. Re-Query NIM with ICL Prompts

Each test question now includes the 5 curated examples in the prompt, teaching the model the expected answer format and domain knowledge.

In [ ]:
# ============================================================
# Format ICL Prompts — From NVIDIA DLI Course Lab
# ============================================================

icl_prompts = []
for _, row in test_df.iterrows():
    prompt = format_finance_prompt(
        question=row["question"],
        context=row.get("context", ""),
        evidence=row.get("evidence", ""),
        icl_examples=icl_examples,  # 5-shot ICL
    )
    icl_prompts.append(prompt)

print(f"Formatted {len(icl_prompts)} ICL prompts")
print(f"Average prompt length: {np.mean([len(p) for p in icl_prompts]):.0f} chars")
print(f"\n--- Sample ICL Prompt (first 800 chars) ---")
print(icl_prompts[0][:800])
print("...")

In [ ]:
# ============================================================
# Batch Query NIM with ICL — From NVIDIA DLI Course Lab
# ============================================================

nim_client = NIMInferenceClient(model="meta/llama-3.1-8b-instruct")

print("Starting ICL (5-shot) inference via NVIDIA NIM...")
start_time = time.time()

icl_responses = nim_client.batch_query(
    prompts=icl_prompts,
    system_prompt=(
        "You are a precise financial analyst specializing in SEC filings. "
        "Answer questions accurately and concisely based on the provided context. "
        "Follow the format shown in the examples."
    ),
    temperature=0.1,
    max_tokens=512,
    delay=0.5,
)

elapsed = time.time() - start_time
print(f"\nICL inference complete in {elapsed:.1f}s")

## 3. Evaluate ICL Responses

In [ ]:
# ============================================================
# Automated Metrics — ICL
# ============================================================

predictions_icl = icl_responses
references = test_df["answer"].tolist()

icl_auto_metrics = {
    "exact_match": compute_exact_match(predictions_icl, references),
    "f1_score": compute_f1_score(predictions_icl, references),
    "avg_prediction_length_words": np.mean([len(p.split()) for p in predictions_icl]),
}

print_metrics_summary(icl_auto_metrics, "ICL (5-shot) — Automated Metrics")

In [ ]:
# ============================================================
# LLM-as-a-Judge — ICL — From NVIDIA DLI Course Lab
# ============================================================

judge = LLMJudge(client=nim_client, judge_model="meta/llama-3.1-70b-instruct")

print("Running LLM-as-a-Judge on ICL responses...")
icl_judge_results = judge.evaluate_batch(
    questions=test_df["question"].tolist(),
    predictions=icl_responses,
    references=references,
    evidences=test_df.get("evidence", pd.Series([""] * len(test_df))).tolist(),
    criteria=["correctness", "faithfulness", "conciseness"],
)

# Aggregate
icl_judge_metrics = {}
for criterion in ["correctness", "faithfulness", "conciseness"]:
    col = f"{criterion}_score"
    if col in icl_judge_results.columns:
        icl_judge_metrics[f"judge_{criterion}_mean"] = icl_judge_results[col].mean()

score_cols = [c for c in icl_judge_results.columns if c.endswith("_score")]
if score_cols:
    icl_judge_results["avg_judge_score"] = icl_judge_results[score_cols].mean(axis=1)
    icl_judge_metrics["judge_avg_score"] = icl_judge_results["avg_judge_score"].mean()

print_metrics_summary(icl_judge_metrics, "ICL (5-shot) — Judge Metrics")

## 4. Compare ICL vs Baseline

In [ ]:
# ============================================================
# Side-by-Side Comparison — From NVIDIA DLI Course Lab
# ============================================================

baseline_metrics = load_results("baseline_metrics.json")

all_icl_metrics = {**icl_auto_metrics, **icl_judge_metrics}

# Comparison table
comparison = pd.DataFrame({
    "Metric": list(all_icl_metrics.keys()),
    "Baseline": [baseline_metrics.get(k, 0) for k in all_icl_metrics.keys()],
    "ICL (5-shot)": list(all_icl_metrics.values()),
})
comparison["Improvement"] = comparison.apply(
    lambda row: f"+{((row['ICL (5-shot)'] - row['Baseline']) / max(row['Baseline'], 0.001)) * 100:.1f}%"
    if row['Baseline'] > 0 else "N/A",
    axis=1
)

print("\n=== ICL vs Baseline Comparison ===")
print(comparison.to_string(index=False))

# Save
save_results(icl_judge_results, "icl_judge_results.csv")
save_results(all_icl_metrics, "icl_metrics.json")

# Save ICL responses
icl_results_df = test_df.copy()
icl_results_df["model_response"] = icl_responses
icl_results_df["model_config"] = "icl_5shot_llama31_8b"
save_results(icl_results_df, "icl_responses.csv")

In [ ]:
# ============================================================
# Log ICL to MLflow
# ============================================================

log_metrics_to_mlflow(
    metrics=all_icl_metrics,
    run_name="icl_5shot_llama31_8b",
    experiment_name="financebench-llm",
    params={
        "model": "meta/llama-3.1-8b-instruct",
        "method": "icl_5shot",
        "n_icl_examples": 5,
        "dataset": "PatronusAI/financebench",
    },
)

print("\n✓ Notebook 3 complete — proceed to Notebook 4 for LoRA fine-tuning.")